In [2]:
import cv2
import glob
import numpy as np
import pandas as pd
from ultralytics          import YOLO
from sklearn.svm          import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.metrics        import classification_report, accuracy_score
from sklearn.pipeline       import make_pipeline

BASE_DIR  = "."
INPUT_DIR = f"{BASE_DIR}/dataset"

KELAS = {
    # Gerakan mendorong
    "PUSH"    : "Push",
    
    # Gerakan menarik
    "PULL"    : "Pull",
    
    # Gerakan Kaki
    "LEG"     : "Leg"
}

yolo = YOLO('yolov8n-pose.pt')

In [3]:
print("TAHAP 1: Ekstraksi Keypoint dari Dataset")

X, y = [], []

for folder, label in KELAS.items():
    images  = glob.glob(f"{INPUT_DIR}/{folder}/**/*.jpg", recursive=True)
    images += glob.glob(f"{INPUT_DIR}/{folder}/**/*.png", recursive=True)
    print(f"\n[{label}] {len(images)} gambar ditemukan")


    for path in images:
        img = cv2.imread(path)
        if img is None:
            continue

        results = yolo(path, verbose=False)
        
        if results[0].keypoints is None or len(results[0].keypoints.xy) == 0:
            print(f"  [SKIP] Kaga ada orang di gambar: {path}")
            continue

        kp = results[0].keypoints.xyn[0].cpu().numpy()
        coords = kp.flatten().tolist()
        
        

        if len(coords) == 34:   # 17 keypoints × 2
            X.append(coords)
            y.append(label)

print(f"\nTotal data terkumpul: {len(X)}")
print(pd.Series(y).value_counts().to_string())

TAHAP 1: Ekstraksi Keypoint dari Dataset

[Push] 641 gambar ditemukan
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_32.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_33.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_62.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_63.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_65.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_66.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_70.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\bench\bench_95.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\pushup\pushup2_frame_0_jpg.rf.093163565e9c281a4f283a7bf3a4e649.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\pushup\pushup2_frame_100_jpg.rf.a280a7cc2754403a492785040579fced.jpg
  [SKIP] Kaga ada orang di gambar: ./dataset/PUSH\pushup\pushup2_frame_105_jpg.rf.e48ea5e458ce67a17bc982f462cacbf6.jpg
  [SKIP] Kaga ada orang

In [4]:
print("TAHAP 2: Training & Evaluasi Model")

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    stratify     = y,
    random_state = 42
)

# SVM
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, class_weight="balanced"))
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
svm_acc  = accuracy_score(y_test, svm_pred)

# MLP
mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(128, 64),
                                  max_iter=500, random_state=42))
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)
mlp_acc  = accuracy_score(y_test, mlp_pred)

# Hasil
print(f"\nSVM")
print(f"Akurasi: {svm_acc*100:.2f}%")
print(classification_report(y_test, svm_pred))

print(f"\nMLP")
print(f"Akurasi: {mlp_acc*100:.2f}%")
print(classification_report(y_test, mlp_pred))

pemenang = "SVM" if svm_acc > mlp_acc else "MLP"
print(f"\n{'='*50}")
print(f"  PEMENANG: {pemenang}")
print(f"  SVM : {svm_acc*100:.2f}%")
print(f"  MLP : {mlp_acc*100:.2f}%")

TAHAP 2: Training & Evaluasi Model

SVM
Akurasi: 94.34%
              precision    recall  f1-score   support

         Leg       0.92      0.93      0.93       118
        Pull       0.94      0.95      0.95       199
        Push       0.96      0.94      0.95       125

    accuracy                           0.94       442
   macro avg       0.94      0.94      0.94       442
weighted avg       0.94      0.94      0.94       442


MLP
Akurasi: 97.29%
              precision    recall  f1-score   support

         Leg       0.96      0.96      0.96       118
        Pull       0.98      0.98      0.98       199
        Push       0.98      0.98      0.98       125

    accuracy                           0.97       442
   macro avg       0.97      0.97      0.97       442
weighted avg       0.97      0.97      0.97       442


  PEMENANG: MLP
  SVM : 94.34%
  MLP : 97.29%


In [38]:
print("TAHAP 3: Live Demo Klasifikasi")

WARNA = {
    "Push": (0, 255,   0),
    "Pull": (255, 165, 0),
    "Leg" : (0,   0, 255),
}

def demo(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Gagal memuat: {image_path}")
        return

    results = yolo(image_path, verbose=False)
    if results[0].keypoints is None:
        print("Pose tidak terdeteksi!")
        return

    coords   = results[0].keypoints.xyn[0].cpu().numpy().flatten().tolist()
    pred_svm = svm.predict([coords])[0]
    pred_mlp = mlp.predict([coords])[0]

    annotated     = results[0].plot()
    tinggi, lebar = annotated.shape[:2]
    font_size     = max(0.7, lebar / 700.0)
    tebal         = max(2, int(font_size * 2.5))
    margin        = int(lebar * 0.03)

    for i, (teks, warna) in enumerate([
        (f"SVM : {pred_svm}", WARNA[pred_svm]),
        (f"MLP : {pred_mlp}", WARNA[pred_mlp])
    ]):
        pos_y = int(tinggi * (0.08 + i * 0.09))
        cv2.putText(annotated, teks, (margin, pos_y),
                    cv2.FONT_HERSHEY_SIMPLEX, font_size, warna, tebal)

    cv2.imshow("Live Demo PPL Classifier", annotated)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print(f"SVM → {pred_svm} | MLP → {pred_mlp}")

demo(f"{BASE_DIR}/test/legcuy.jpg")


TAHAP 3: Live Demo Klasifikasi
SVM → Leg | MLP → Push
